### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="electric_motor_temperature_prediction_1m",
    version_from_unique_name="electric_motor_temperature_prediction",
    version_comment="""
We build the recommended single train/test split (250k test rows) and sub-sample it to at most 1M train and 250k test rows with `curation_recommendations.subsample_split_to_budget`, keeping whole profiles.
""",
    dataset_year="2021",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature",
    download_description="""
kaggle datasets download wkirgsn/electric-motor-temperature --unzip \
&& mkdir -p local-data-warehouse/electric_motor_temperature_prediction \
&& mv measures_v2.csv local-data-warehouse/electric_motor_temperature_prediction/
""",
    # References
    academic_reference_bibtex="""@article{kirchgassner2020estimating,
  title={Estimating electric motor temperatures with deep residual machine learning},
  author={Kirchg{\"a}ssner, Wilhelm and Wallscheid, Oliver and B{\"o}cker, Joachim},
  journal={IEEE Transactions on Power Electronics},
  volume={36},
  number={7},
  pages={7480--7488},
  year={2020},
  publisher={IEEE}
}
""",
    academic_reference_bibtex_key="kirchgassner2020estimating",
    license="CC BY-SA 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We select the task to predict permanent magnet temperature from the input features. We create grouped splits on the profile_id, which corresponds to predicting the temperature for an unseen motor run session, thus simulating how the model would be used in reality.

- The original study also incorporates temporal connection within the session. We re-create a time-index (https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature/discussion/147446).
- We also add the derived inputs used in the original study.
- We also add the EWMA/WEMS features with window sizes of [500, 2000, 4000, 8000]. To avoid the problem of having an incorrect EWMA/WEMS state and since not all recording are warmed up (https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature/discussion/117319), we drop the first 500 samples (first span) of each profile to simulate such a warm-up.
"""
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="permanent_magnet_temperature",
    problem_type="regression",
    objective_metric_name="rmse",
    # For grouped data
    group_on="profile_id",
    group_labels="per_sample",
    group_time_on="profile_time_index"
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "measures_v2.csv")
print("Loaded data shape:", df.shape)

# Add time feature
df["profile_time_index"] = df.groupby("profile_id").cumcount() + 1

df = df.rename(columns={
    "pm": "permanent_magnet_temperature",
})

df = df.drop(columns=[
    # Other targets
    "stator_winding",
    "stator_tooth",
    "stator_yoke",
    # excluded as in the original study
    "torque",
])

# Add derived features  (from Table II)
# Voltage magnitude
df["u_s"] = np.sqrt(df["u_d"]**2 + df["u_q"]**2)
# Current magnitude
df["i_s"] = np.sqrt(df["i_d"]**2 + df["i_q"]**2)
# Apparent power
df["S_el"] = df["u_s"] * df["i_s"]
# Joint interaction terms
df["i_s_omega"] = df["i_s"] * df["motor_speed"]
df["S_el_omega"] = df["S_el"] * df["motor_speed"]


def add_grouped_ewm_features(
    input_df: pd.DataFrame,
    group_col: str,
    time_col: str,
    cols: list[str],
    input_spans: list[int],
) -> pd.DataFrame:
    input_df = input_df.sort_values([group_col, time_col]).copy()

    for col in cols:
        for span in input_spans:
            alpha = 2 / (span + 1)

            # EWMA
            input_df[f"{col}_ewma_{span}"] = (
                input_df.groupby(group_col, sort=False)[col]
                  .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
            )

            # EWMS
            input_df[f"{col}_ewms_{span}"] = (
                input_df.groupby(group_col, sort=False)[col]
                  .transform(lambda s: s.ewm(alpha=alpha, adjust=False).std())
            )

    return input_df

input_cols = [
    # Original features
    "u_d", "u_q",
    "i_d", "i_q",
    "coolant", "motor_speed", "ambient",
    # Derived features
    "u_s", "i_s", "S_el", "i_s_omega", "S_el_omega",
]
spans = [500, 2000, 4000, 8000]
df = add_grouped_ewm_features(
    input_df=df,
    group_col="profile_id",
    time_col="profile_time_index",
    cols=input_cols,
    input_spans=spans,
)

# Drop the first 500 samples of each profile to simulate warm-up and avoid incorrect EWMA/WEMS state
df = (
    df.sort_values(["profile_id", "profile_time_index"])
      .loc[lambda d: d.groupby("profile_id").cumcount() >= 500]
      .reset_index(drop=True)
)
df["profile_id"] = df["profile_id"].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle data

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

# Sub-sample the single train/test split to the row budget (1M train / 250k test rows), keeping whole profiles.
train_idx, test_idx = splits[0][0]
df, train_idx, test_idx = curation_recommendations.subsample_split_to_budget(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    group_on=task_mold.group_on,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)
splits = {0: {0: (train_idx, test_idx)}}
print(f"Final train size: {len(train_idx):,}, final test size: {len(test_idx):,}")
print(f"Final train groups: {df.iloc[train_idx][task_mold.group_on].nunique():,}, final test groups: {df.iloc[test_idx][task_mold.group_on].nunique():,}")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Recommended single grouped train/test split, sub-sampled to the row budget of 1M train and 250k test rows, keeping whole profiles.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()